# Machine Learning Pipeline

In [ ]:
import os
import sys
import itertools
import pickle
import keras
import mlflow
import mlflow.keras
import mlflow.pyfunc
import tensorflow as tf
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from src import config
from tqdm import tqdm
from pathlib import Path
from keras import layers, optimizers
from mlflow import MlflowClient
from mlflow.models.signature import infer_signature
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import MinMaxScaler

## 1. Parameter Global


In [ ]:
RANDOM_STATE = 42
TICKER = "BBRI.JK"
TARGET_COL = "close"

# Hyperparameter
TIME_STEPS = [10, 20]
OPTIMIZERS = ["RMSprop"]
BATCH_SIZES = [32]
LEARNING_RATES = [0.0001]
MODELS = ["LSTM", "GRU"]

EPOCHS = 50
TRAIN_SIZE = 0.90

In [ ]:
keras.utils.set_random_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

In [ ]:
if config.MLFLOW_TRACKING_URI:
    mlflow.set_tracking_uri(config.MLFLOW_TRACKING_URI)

mlflow.set_experiment(config.get_experiment_name(TICKER))

## 2. Data Understanding


### 2.1. Memuat Data


In [ ]:
df = pd.read_parquet(config.get_data_path(TICKER))
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

print(f"Periode Observasi : {df['date'].min().date()} s/d {df['date'].max().date()}")
print(f"Total Baris Data  : {len(df)}")

### 2.2. Pengecekan Gap Tanggal


In [ ]:
full_date_range = pd.date_range(start=df["date"].min(), end=df["date"].max(), freq="D")
missing_dates = full_date_range.difference(df["date"])

print(f"Jumlah Gap Tanggal : {len(missing_dates)}")

### 2.3. Visualisasi Harga Historis


In [ ]:
plt.figure(figsize=(13, 5))
sns.lineplot(data=df, x="date", y=TARGET_COL)
plt.xlabel("Tanggal")
plt.ylabel("Harga (Rp)")
plt.title(f"Pergerakan Harga Historis - {TICKER}")
plt.tight_layout()
plt.show()
plt.close()

## 3. Data Preparation


### 3.2. Pembagian Data


In [ ]:
train_len = int(len(df) * TRAIN_SIZE)
train_df = df.iloc[:train_len].copy().reset_index(drop=True)
test_df = df.iloc[train_len:].copy().reset_index(drop=True)

print(f"Jumlah Data Latih ({int(TRAIN_SIZE*100)}%)     : {len(train_df)}")
print(f"Jumlah Data Uji ({int((1-TRAIN_SIZE)*100)}%)   : {len(test_df)}")

### 3.3. Normalisasi Skala


In [ ]:
scaler = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler.fit_transform(train_df[[TARGET_COL]].values)
test_scaled = scaler.transform(test_df[[TARGET_COL]].values)

scaler_path = config.get_scaler_path(TICKER)
with open(scaler_path, "wb") as f:
    pickle.dump(scaler, f)

print(f"Min Scaler (Train)  : {scaler.data_min_[0]:.2f}")
print(f"Max Scaler (Train)  : {scaler.data_max_[0]:.2f}")
print(f"Scaler tersimpan di : {scaler_path}")

In [ ]:
def create_windowed_dataset(train_arr, test_arr, time_steps, batch_size):
    ds_train = keras.utils.timeseries_dataset_from_array(
        data=train_arr[:-1],
        targets=train_arr[time_steps:, 0],
        sequence_length=time_steps,
        batch_size=batch_size,
        shuffle=True,
        seed=RANDOM_STATE,
    )

    test_inputs = np.concatenate([train_arr[-time_steps:], test_arr], axis=0)

    ds_test = keras.utils.timeseries_dataset_from_array(
        data=test_inputs[:-1],
        targets=test_inputs[time_steps:, 0],
        sequence_length=time_steps,
        batch_size=batch_size,
        shuffle=False,
    )

    return ds_train, ds_test

## 4. Model & Evaluasi


### 4.1. Arsitektur Model

In [ ]:
def build_model(model_type, time_steps, opt_name, lr):
    keras.backend.clear_session()

    if opt_name == "SGD":
        opt = optimizers.SGD(learning_rate=lr)
    elif opt_name == "Adam":
        opt = optimizers.Adam(learning_rate=lr)
    elif opt_name == "RMSprop":
        opt = optimizers.RMSprop(learning_rate=lr)
    else:
        raise ValueError(f"Optimizer {opt_name} tidak didukung.")

    model = keras.Sequential()
    model.add(layers.Input(shape=(time_steps, 1)))

    if model_type == "LSTM":
        model.add(layers.LSTM(units=50, return_sequences=False))
    elif model_type == "GRU":
        model.add(layers.GRU(units=50, return_sequences=False))
    else:
        raise ValueError(f"Tipe model {model_type} tidak valid.")

    model.add(layers.Dense(units=1))
    model.compile(optimizer=opt, loss="mean_squared_error")
    
    return model

In [ ]:
def calculate_metrics(y_true, y_pred):
    mse = float(mean_squared_error(y_true, y_pred))
    rmse = float(np.sqrt(mse))
    mape = float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100)

    return {
        "MSE": mse,
        "RMSE": rmse,
        "MAPE": mape
    }

### 4.2. Eksperimen Model

In [ ]:
def execute_trial(trial_params, train_scaled, test_scaled, y_actual, scaler, epochs):
    ts, opt, bs, lr, m_type = trial_params

    ds_train, ds_test = create_windowed_dataset(
        train_arr=train_scaled,
        test_arr=test_scaled,
        time_steps=ts,
        batch_size=bs,
    )

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="loss",
            patience=8,
            restore_best_weights=True,
        )
    ]

    model = build_model(model_type=m_type, time_steps=ts, opt_name=opt, lr=lr)
    history = model.fit(
        ds_train,
        epochs=epochs,
        shuffle=False,
        callbacks=callbacks,
        verbose=0,
    )

    preds_scaled = model.predict(ds_test, verbose=0)
    preds_inv = scaler.inverse_transform(preds_scaled).flatten()
    metrics = calculate_metrics(y_true=y_actual, y_pred=preds_inv)

    record = {
        "model": m_type,
        "time_steps": ts,
        "optimizer": opt,
        "batch_size": bs,
        "learning_rate": lr,
        "epochs_trained": len(history.epoch),
        **metrics,
    }

    return record, model

In [ ]:
param_combinations = list(
    itertools.product(TIME_STEPS, OPTIMIZERS, BATCH_SIZES, LEARNING_RATES, MODELS)
)

hyperparameter_results = []
y_actual = test_df[TARGET_COL].values
progress_bar = tqdm(param_combinations, desc="Grid Search", unit="trial")

best_models_data = {
    "LSTM": {"rmse": float("inf"), "record": None, "model": None},
    "GRU": {"rmse": float("inf"), "record": None, "model": None},
}

run_name = f"{TICKER}_Hyperparameter"

with mlflow.start_run(run_name=run_name) as parent_run:
    mlflow.set_tags({
        "target": TARGET_COL,
        "stage": "hyperparameter",
    })

    dataset_source = mlflow.data.from_pandas(df, targets=TARGET_COL, name=f"{TICKER}_data")
    mlflow.log_input(dataset_source, context="training_and_testing")

    for params in progress_bar:
        m_ts, m_opt, m_bs, m_lr, m_name = params

        record, trained_model = execute_trial(
            trial_params=params,
            train_scaled=train_scaled,
            test_scaled=test_scaled,
            y_actual=y_actual,
            scaler=scaler,
            epochs=EPOCHS,
        )

        hyperparameter_results.append(record)
        current_rmse = record["RMSE"]

        if current_rmse < best_models_data[m_name]["rmse"]:
            best_models_data[m_name]["rmse"] = current_rmse
            best_models_data[m_name]["record"] = record
            best_models_data[m_name]["model"] = trained_model

            save_path = config.get_model_path(TICKER, m_name)
            trained_model.save(save_path)

        progress_bar.set_postfix({
            "Model": m_name,
            "RMSE": f"{current_rmse:.2f}",
            "LSTM_Best": f"{best_models_data['LSTM']['rmse']:.2f}",
            "GRU_Best": f"{best_models_data['GRU']['rmse']:.2f}",
        })

    # Simpan riwayat hyperparameter
    hyperparameter_csv_path = config.get_hyperparameter_path(TICKER)
    hyperparameter_results_df = pd.DataFrame(hyperparameter_results)
    hyperparameter_results_df.to_csv(hyperparameter_csv_path, index=False)

    mlflow.log_artifact(str(hyperparameter_csv_path), artifact_path="hyperparameter")
    mlflow.log_metrics({
        "LSTM_RMSE": float(best_models_data["LSTM"]["rmse"]),
        "GRU_RMSE": float(best_models_data["GRU"]["rmse"]),
    })

### 4.3. Penentuan Parameter Terbaik

In [ ]:
best_indices = hyperparameter_results_df.groupby("model")["RMSE"].idxmin().values
best_configs_df = hyperparameter_results_df.loc[best_indices].reset_index(drop=True)

display(best_configs_df)

### 4.4. Inferensi Model

In [ ]:
for _, row in best_configs_df.iterrows():
    cfg = row.to_dict()
    model_name = str(cfg["model"])
    ts = int(cfg["time_steps"])
    bs = int(cfg["batch_size"])

    # Muat bobot model terbaik
    model_path = config.get_model_path(TICKER, model_name)
    loaded_model = keras.models.load_model(model_path)

    # Siapkan windowed dataset untuk data uji
    _, ds_test = create_windowed_dataset(train_scaled, test_scaled, ts, bs)

    # Prediksi untuk data uji
    pred_test = scaler.inverse_transform(loaded_model.predict(ds_test, verbose=0)).flatten()

    # Visualisasi data latih, data uji, dan prediksi data uji
    fig, ax = plt.subplots(figsize=(13, 5))
    ax.plot(train_df["date"].iloc[ts:], train_df[TARGET_COL].iloc[ts:], label="Train Actual", color="tab:blue")
    ax.plot(test_df["date"], test_df[TARGET_COL], label="Test Actual", color="tab:green")
    ax.plot(test_df["date"], pred_test, label=f"{model_name} Prediction (Test)", color="tab:red", linestyle="--")

    ax.set_title(f"Inferensi {model_name} - {TICKER} (RMSE: {cfg['RMSE']:.2f}, MAPE: {cfg['MAPE']:.2f}%)")
    ax.set_xlabel("Tanggal")
    ax.set_ylabel("Harga (Rp)")
    ax.legend()
    plt.tight_layout()
    plt.show()
    plt.close(fig)

### 4.5. Registrasi Model

In [ ]:
client = MlflowClient()

for _, row in best_configs_df.iterrows():
    cfg = row.to_dict()
    model_name = str(cfg["model"])
    ts = int(cfg["time_steps"])
    bs = int(cfg["batch_size"])

    # Muat bobot model terbaik dari disk lokal
    model_path = config.get_model_path(TICKER, model_name)
    loaded_model = keras.models.load_model(model_path)

    # Inisialisasi Signature Tensor
    sample_x = np.zeros((1, ts, 1), dtype=np.float32)
    sample_pred = loaded_model.predict(sample_x, verbose=0)
    signature = infer_signature(sample_x, sample_pred)

    reg_model_name = f"{TICKER}_{model_name}"
    run_name = f"{TICKER}_{model_name}_Registration"

    with mlflow.start_run(run_name=run_name):
        # Metadata Tags
        mlflow.set_tags({
            "target": TARGET_COL,
            "stage": "registration",
        })

        mlflow.log_params(cfg)
        mlflow.log_metrics({
            "RMSE": float(cfg["RMSE"]),
            "MAPE": float(cfg["MAPE"]),
            "MSE": float(cfg["MSE"]),
        })

        # Bundle file scaler ke dalam artefak
        mlflow.log_artifact(str(scaler_path), artifact_path="scaler")

        #  Daftarkan model Keras 
        mlflow.keras.log_model(
            model=loaded_model,
            name=f"model_{model_name.lower()}",
            registered_model_name=reg_model_name,
            signature=signature,
        )

        # Ambil nomor versi terbaru dan tetapkan alias @challenger
        latest_model = client.get_registered_model(reg_model_name)
        model_version = latest_model.latest_versions[0].version

        client.set_registered_model_alias(
            name=reg_model_name,
            alias="challenger",
            version=model_version,
        )

        print(f"Model {reg_model_name} (v{model_version}) berhasil didaftarkan bersama scalernya.")
        print(f"-> URI Evaluasi: models:/{reg_model_name}@challenger")